# 03 — Feature Engineering

## Purpose

This notebook rebuilds the model-ready dataset from the cleaned World Bank
panel. Every step is a deliberate design decision, documented here so the
pipeline is reproducible and auditable.

## Data Preparation Pipeline

**`clean_wdi_eu27_1995_2025.csv`**  
837 rows · 27 countries · 1995–2025

↓

**Add lags (`lag_1`, `lag_2`)**  
Previous years' GDP growth, calculated within each country

↓

**Add target (next-year growth)**  
`shift(-1)` within each country

↓

**Drop rows without target**  
2025 has no next-year GDP growth value

↓

**Add GDP-per-capita groups**  
Tertiles based on each country's mean GDP per capita

↓

**Add chronological split**  
Train: 1995–2019  
Test: 2020–2024

↓

**Impute FDI (country median)**  
6 missing rows, all Luxembourg, 1996–2001

↓

**`model_data_v2.csv`**  
810 rows · 19 columns


## Why a rebuild?

An earlier version of the modeling file (`model_data_before_imputation.csv`)
contained a bug: `gdp_growth` and `gdp_growth_lag_1` were identical in all
783 rows. A lag column should hold the *previous* year's value, not the
current year's — so the file was not giving the model a genuine "last
year's growth" signal.

This notebook rebuilds the dataset with correct lags. The methodology is
otherwise unchanged: same countries, same years, same chronological split.

## Reproducibility

All logic lives in `src/preprocessing.py`. This notebook imports the
pipeline and displays the verification results. Random seed is set in
`src/config.py` (SEED = 8, aliased to `RANDOM_STATE`).

In [8]:
import sys
from pathlib import Path
# Make the project root importable so `from src import ...` works
current = Path.cwd()
project_root = current.parent if current.name == "notebooks" else current
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [9]:
import pandas as pd
from src import config, preprocessing as pp

## Setup

We add the project root to `sys.path` so that `from src import ...` works
from inside the `notebooks/` folder. Then we import the pipeline and run
it. `build_model_dataset()` executes all seven steps in sequence and
writes the output to `data/processed/model_data_v2.csv`.

In [10]:
df = pp.build_model_dataset()

Dropped 27 rows with no target. Remaining: 810
Saved 810 rows to /home/ricci-preto/IronHack/Classes/ML Project/data/processed/model_data_v2.csv


## Verification

We verify that the rebuild did what it was supposed to:

1. **Shape and columns** — expected 810 rows × 19 columns.
2. **Year range and country count** — 1995–2024, 27 countries, 30 rows each.
3. **Chronological split** — train 1995–2019, test 2020–2024, no overlap.
4. **Missing values** — expected only in the lag columns at the start of
   the panel (1995 for `lag_1`, 1995–1996 for `lag_2`).
5. **Row-by-row verification on Germany** — confirms the shift chain is
   correct within each country.

In [11]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print()
print("Year range:", df["Year"].min(), "–", df["Year"].max())
print("Countries:", df["Country Code"].nunique())
print()

# Rows per country — all should be equal
print("Rows per country:")
print(df["Country Code"].value_counts().value_counts())
print()

# Split
print("Split counts:")
print(df["dataset_split"].value_counts())
print()
print("Year range per split:")
print(df.groupby("dataset_split")["Year"].agg(["min", "max", "count"]))
print()

# Missing values
print("Missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])
print()

# Look at one country to verify the shift logic
print("Germany, first 8 rows sorted by year:")
cols_to_show = ["Year", "gdp_growth", "gdp_growth_lag_1", "gdp_growth_lag_2", "target_gdp_growth_next_year"]
print(df[df["Country Code"] == "DEU"][cols_to_show].head(8).to_string(index=False))
print()

# Look at last 5 rows for Germany — check the tail
print("Germany, last 8 rows:")
print(df[df["Country Code"] == "DEU"][cols_to_show].tail(8).to_string(index=False))

Shape: (810, 19)
Columns: ['Country Code', 'Country Name', 'Year', 'fdi_inflows_percent_gdp', 'inflation', 'government_consumption_percent_gdp', 'exports_percent_gdp', 'gross_fixed_capital_formation', 'imports_percent_gdp', 'gdp_growth', 'gdp_per_capita', 'unemployment', 'population_growth', 'trade_openness', 'gdp_growth_lag_1', 'gdp_growth_lag_2', 'target_gdp_growth_next_year', 'gdp_per_capita_group', 'dataset_split']

Year range: 1995 – 2024
Countries: 27

Rows per country:
count
30    27
Name: count, dtype: int64

Split counts:
dataset_split
train    675
test     135
Name: count, dtype: int64

Year range per split:
                min   max  count
dataset_split                   
test           2020  2024    135
train          1995  2019    675

Missing values per column:
gdp_growth_lag_1    27
gdp_growth_lag_2    54
dtype: int64

Germany, first 8 rows sorted by year:
 Year  gdp_growth  gdp_growth_lag_1  gdp_growth_lag_2  target_gdp_growth_next_year
 1995    1.504994               N

## Verification — result

Every check passes as expected.

### Shape and structure
- **810 rows × 19 columns** — 27 countries × 30 years (1995–2024).
- Every country has exactly **30 rows**.

### Chronological split
| Split | Years | Rows |
|---|---|---|
| Train | 1995–2019 | 675 |
| Test | 2020–2024 | 135 |

The split is strictly chronological — the model never sees future years
during training. This is essential for time-series forecasting, and the
main reason we avoided random train/test splitting.

### Missing values
- `gdp_growth_lag_1`: **27 NaNs** (one per country, in 1995)
- `gdp_growth_lag_2`: **54 NaNs** (one per country, in 1995 and 1996)
- FDI: **0 NaNs** — imputation worked
- Target: **0 NaNs** — 2025 was correctly dropped

These are the expected NaN patterns: the first one or two years of each
country's panel have no prior values to shift from. Downstream models
handle these NaNs differently — XGBoost and Random Forest accept them
natively; Linear Regression uses `SimpleImputer(strategy="median")`
inside a pipeline (see `src/models.py`).

### Germany row-level check

The Germany table confirms the shift chain is correct:

| Year | gdp_growth | lag_1 | lag_2 | target |
|---|---|---|---|---|
| 1995 | 1.504994 | NaN | NaN | 1.037876 |
| 1996 | 1.037876 | 1.504994 | NaN | 1.854322 |
| 1997 | 1.854322 | 1.037876 | 1.504994 | 2.095612 |
| 1998 | 2.095612 | 1.854322 | 1.037876 | 2.129570 |

- 1996's `lag_1` equals 1995's `gdp_growth` ✅
- 1996's `lag_2` is NaN (no 1994 to draw from) ✅
- 1997's `lag_2` equals 1995's `gdp_growth` ✅
- Each row's `target` equals the **next** year's `gdp_growth` ✅
- All three growth columns are distinct — no duplication.

Look at 2019 → 2020 → 2021 in the tail of the table:
the COVID crash (−4.13% in 2020) and rebound (+3.91% in 2021) are both
captured correctly. The model will see 2019 indicators to predict −4.13%
in 2020 (which is a test-set year, so it can't memorize it).

### Conclusion

The lag features now carry genuine time-shifted information. The rebuild
fixed the duplicate-lag bug from the earlier file and preserved every
other methodological choice.